# Week 2: Baseline

A pretrained ResNet18 with every layer frozen except a new 4-output head is trained on the frozen split, with no augmentation, class-weighted loss, for 10 epochs. It serves as a baseline and later experiments are compared with its score. Only train and validation sets are used.

** Runtime > Change runtime type > GPU

## 1. Setup: mount Drive, unzip data, check the split

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json, random, time
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

PROJECT = "/content/drive/MyDrive/fruit_project"
DATA = "/content/dataset"
if not os.path.isdir(DATA):
    shutil.unpack_archive(f"{PROJECT}/dataset.zip", DATA)     # takes a minute or two

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
assert device.type == "cuda", "Runtime -> Change runtime type -> GPU, then rerun"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

split = json.load(open(f"{PROJECT}/split.json"))
probe = datasets.ImageFolder(DATA)
assert probe.class_to_idx == split["class_to_idx"], "class order changed"
assert len(probe) == sum(len(split[k]) for k in ["train", "val", "test"]), "image count mismatch"
class_names = probe.classes
num_classes = len(class_names)
print("OK:", probe.class_to_idx, len(probe), "images")

## 2. Datasets and loaders (no augmentation)

In [ ]:
# 2. Datasets and loaders
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BASE_TFM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def make_datasets(train_tfm):
    """Returns (train, val). The test set is deliberately not built here."""
    train_full = datasets.ImageFolder(DATA, transform=train_tfm)
    eval_full  = datasets.ImageFolder(DATA, transform=BASE_TFM)
    return Subset(train_full, split["train"]), Subset(eval_full, split["val"])

train_ds, val_ds = make_datasets(BASE_TFM)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

x, y = next(iter(train_loader))
print(x.shape, y[:10])                 # torch.Size([32, 3, 224, 224]) and labels 0..3
print(len(train_ds), len(val_ds))      # 2236 480

## 3. Class weights

In [ ]:
# 3. Class weights (computed from the training split only)
train_targets = np.array(probe.targets)[split["train"]]
counts = np.bincount(train_targets, minlength=num_classes)
weights = len(train_targets) / (num_classes * counts)
for n, c, w in zip(class_names, counts, weights):
    print(f"{n:15s} {c:5d} images  weight {w:.2f}")

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

## 4. Model and training

In [ ]:
# 4. Model and training
def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False                                   # freeze the backbone
    model.fc = nn.Linear(model.fc.in_features, num_classes)       # new trainable head
    return model.to(device)

def run_epoch(model, loader, criterion, optimizer=None):
    """Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

model = build_model(num_classes)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))  # 2052
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

os.makedirs(f"{PROJECT}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT}/results", exist_ok=True)
CKPT = f"{PROJECT}/checkpoints/baseline_frozen.pt"

EPOCHS = 10
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader, criterion)
    for k, v in zip(history, (tl, ta, vl, va)):
        history[k].append(v)
    flag = ""
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save({"model_state_dict": model.state_dict(), "class_names": class_names,
                    "epoch": epoch, "val_loss": vl, "val_acc": va}, CKPT)
        flag = "  <- best so far, saved"
    print(f"epoch {epoch:2d} | train loss {tl:.3f} acc {ta:.3f} | "
          f"val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.0f}s{flag}")

## 5. Curves, validation metrics, results log

In [ ]:
# 5. Curves, validation metrics, results log
from sklearn.metrics import classification_report, f1_score
import csv

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.savefig(f"{PROJECT}/results/baseline_frozen_curves.png", dpi=150)
plt.show()

# The validation number comes from the BEST epoch, not the last one
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in val_loader:
        ps.extend(model(x.to(device)).argmax(1).cpu().numpy())
        ys.extend(y.numpy())
ys, ps = np.array(ys), np.array(ps)
val_acc = (ys == ps).mean()
macro_f1 = f1_score(ys, ps, average="macro")

print(classification_report(ys, ps, target_names=class_names, digits=3))
print(f"best epoch {ckpt['epoch']} | val accuracy {val_acc:.3f} | val macro-F1 {macro_f1:.3f}")
print(f"for reference, always guessing Healthy would score {(ys == probe.class_to_idx['Healthy']).mean():.3f}")

path = f"{PROJECT}/results/results.csv"
new = not os.path.exists(path)
with open(path, "a", newline="") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["run", "augmentation", "finetune_mode", "epochs", "val_acc", "val_macro_f1", "notes"])
    w.writerow(["baseline_frozen", "none", "frozen", EPOCHS, round(val_acc, 4), round(macro_f1, 4),
                "class-weighted CE, Adam lr 1e-3, ResNet18"])

## Results

```
Best epoch: 10    Val accuracy: 0.942   Val macro-F1: 0.936
Weakest class (lowest recall): powdery mildew
Notes: always guessing healthy scores 0.490
```